In [14]:
import openseespy.opensees as ops
import math
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d.art3d import Poly3DCollection
plt.rcParams['font.family'] = 'Cambria'
plt.rcParams['figure.dpi'] = 600


In [15]:
# =====================================================================================================================
# BRIDGE INPUT DATA
# =====================================================================================================================
bridge = {

    # --------------------------------------------------------------------------------------------------------------
    # Geometry
    # --------------------------------------------------------------------------------------------------------------
    'geometry': {

        'span'          : 35.0,

        'cross_section' : {
            'footpath_left'  : 1.50,
            'kerb_left'      : 0.45,
            'carriageway_1'  : 4.5,
            'median'         : 0.60,
            'carriageway_2'  : 4.5,
            'kerb_right'     : 0.45,
            'footpath_right' : 1.50
        }
    },

    # --------------------------------------------------------------------------------------------------------------
    # Deck
    # --------------------------------------------------------------------------------------------------------------
    'deck' : {
        'thickness'         : 0.23,
        'overhang'          : 1.25, 
        'wearing_course'    : 0.075
    },

    # --------------------------------------------------------------------------------------------------------------
    # Girders
    # --------------------------------------------------------------------------------------------------------------
    'girders' : {

        'number'    : 5,

        'section'   : {
            'bf_top'    : 0.550,
            'bf_bot'    : 0.650,
            'tf_top'    : 0.025,
            'tf_bot'    : 0.040,
            't_web'     : 0.014,
            'h_web'     : 2.100
        }
    },

    # --------------------------------------------------------------------------------------------------------------
    # Bracing
    # --------------------------------------------------------------------------------------------------------------
    'bracing' : {
        'number'        : 7,
        'area'          : 0.01,
        'configuration' : 'XT'
    },

        # Available Configurations:
            # X   = X diagonals only
            # XT  = X diagonals + top chord
            # XB  = X diagonals + bottom chord
            # XTB = X diagonals + top chord + bottom chord
            # K   = K bracing + bottom chord
            # KT  = K bracing + bottom chord + top chord 

    # --------------------------------------------------------------------------------------------------------------
    # Mesh
    # --------------------------------------------------------------------------------------------------------------
    'mesh' : {
        'length_sub_div_number'  : 25,
        'width_target_mesh_size' : 0.25
    },

    # --------------------------------------------------------------------------------------------------------------
    # Material Properties
    # --------------------------------------------------------------------------------------------------------------
    'materials' : {

        'steel' : {
            'E'     : 2.0e8,
            'nu'    : 0.30,
            'rho'   : 78.5
        },

        'concrete' : {
            'fck'   : 35,
            'nu'    : 0.20,
            'rho'   : 25
        },

        'asphalt' : {
            'rho'   : 22
        }
    },
    
    # --------------------------------------------------------------------------------------------------------------
    # Additional Dead Loads
    # --------------------------------------------------------------------------------------------------------------
    'additional_dead_loads' : {

        'footpath' : {
            'thickness' : 0.150,      # m
            'density'   : 24.0        # kN/m^3
        },

        'kerb' : {
            'thickness' : 0.300,      # m
            'density'   : 24.0        # kN/m^3
        },

        'median' : {
            'thickness' : 0.250,      # m
            'density'   : 24.0        # kN/m^3
        }
    }
}


In [16]:
# =====================================================================================================================
# EXTRACT INPUT DATA
# =====================================================================================================================
# Geometry
length              = bridge['geometry']['span']
cross_section       = bridge['geometry']['cross_section']
footpath_left       = cross_section['footpath_left']
kerb_left           = cross_section['kerb_left']
carriageway_1       = cross_section['carriageway_1']
median_width        = cross_section['median']
carriageway_2       = cross_section['carriageway_2']
kerb_right          = cross_section['kerb_right']
footpath_right      = cross_section['footpath_right']
width               = sum(cross_section.values())

# Deck 
deck_thickness      = bridge['deck']['thickness']
deck_overhang       = bridge['deck']['overhang']
asphalt_thickness   = bridge['deck']['wearing_course']

# Girders
girder_num          = bridge['girders']['number']
bf_top              = bridge['girders']['section']['bf_top']
bf_bot              = bridge['girders']['section']['bf_bot']
tf_top              = bridge['girders']['section']['tf_top']
tf_bot              = bridge['girders']['section']['tf_bot']
t_web               = bridge['girders']['section']['t_web']
h_web               = bridge['girders']['section']['h_web']
girder_depth        =  (tf_top + h_web + tf_bot)

# Bracing
brace_num           = bridge['bracing']['number']
A_brace             = bridge['bracing']['area']
brace_config        = bridge['bracing']['configuration']

# Mesh Parameters
length_sub_div      = bridge['mesh']['length_sub_div_number']
width_mesh_size     = bridge['mesh']['width_target_mesh_size']

# Material Properties
E_steel             = bridge['materials']['steel']['E']         # Modulus of elsticity of steel (kN/m^2) 
Mu_steel            = bridge['materials']['steel']['nu']        # Poisson's ratio of steel
Rho_steel           = bridge['materials']['steel']['rho']       # Unit weight of steel (kN/m^3)
fck_conc            = bridge['materials']['concrete']['fck']    # Characteristic compressive strength of concrete (N/mm^2) 
Mu_conc             = bridge['materials']['concrete']['nu']     # Poisson's ratio of concrete
Rho_conc            = bridge['materials']['concrete']['rho']    # Unit weight of concrete (kN/m^3)
Rho_asphalt         = bridge['materials']['asphalt']['rho']     # Unit weight of asphalt (kN/m^3)

E_conc              = 5000*np.sqrt(fck_conc)*1000               # Modulus of elsticity of concrete (kN/m^2)
G_steel             = E_steel / (2*(1+Mu_steel))                # Modulus of rigidity of steel (kN/m^2)
G_conc              = E_conc / (2*(1+Mu_conc))                  # Modulus of rigidity of concrete (kN/m^2)

# Additional Dead Loads
footpath_thickness  = bridge['additional_dead_loads']['footpath']['thickness']
Rho_footpath        = bridge['additional_dead_loads']['footpath']['density']
kerb_thickness      = bridge['additional_dead_loads']['kerb']['thickness']
Rho_kerb            = bridge['additional_dead_loads']['kerb']['density']
median_thickness    = bridge['additional_dead_loads']['median']['thickness']
Rho_median          = bridge['additional_dead_loads']['median']['density']

# Girder Section Geometry
A_girder = (bf_top*tf_top + bf_bot*tf_bot + t_web*h_web) # Girder cross-sectional area (m^2)

# Neutral axis from bottom of girder (m)
ybar_girder = (bf_bot*tf_bot*(tf_bot/2) + t_web*h_web*(tf_bot + h_web/2)
         + bf_top*tf_top*(tf_bot + h_web + tf_top/2)) / A_girder 

# Girder strong-axis moment of inertia (Izz, m^4)
Izz_girder = (bf_bot*tf_bot**3/12 + bf_bot*tf_bot*(ybar_girder - tf_bot/2)**2 + t_web*h_web**3/12
        + t_web*h_web*(ybar_girder - tf_bot - h_web/2)**2 + bf_top*tf_top**3/12
          + bf_top*tf_top*(tf_bot + h_web + tf_top/2 - ybar_girder)**2)

# Girder weak-axis moment of inertia (Iyy, m^4)
Iyy_girder = (tf_top*bf_top**3/12 + tf_bot*bf_bot**3/12 + h_web*t_web**3/12)

# Torsional constant
J_girder = (bf_top*tf_top**3 + bf_bot*tf_bot**3 + h_web*t_web**3) / 3


In [4]:
# =====================================================================================================================
# MODELLING THE GEOMETRY
# =====================================================================================================================
# Remove any existing model
ops.wipe() 

# Set the model builder - 3 dimensions and 6 degrees of freedom per node
ops.model('basic', '-ndm', 3, '-ndf', 6)

# ---------------------------------------------------------------------------------------------------------------------
# Girder and Bracing Locations
# ---------------------------------------------------------------------------------------------------------------------
girder_location = np.round(np.linspace(deck_overhang, width-deck_overhang, girder_num), 5) # np.linspace(start, end, number_of_points)
brace_location  = np.round(np.linspace(0, length, brace_num), 5) # np.linspace - creates equal spacing btw start and end
girder_spacing  = girder_location[1] - girder_location[0]

print('\n' + '='*80)
print('BRIDGE MODEL')
print('='*80)
print('\n----------------------------------------')
print(f'Girder Spacing = {girder_spacing}')
print()
print('Girder Locations')
print(girder_location)
print()
print('Brace Locations')
print(brace_location)
print('----------------------------------------')


BRIDGE MODEL

----------------------------------------
Girder Spacing = 2.75

Girder Locations
[ 1.25  4.    6.75  9.5  12.25]

Brace Locations
[ 0.       5.83333 11.66667 17.5     23.33333 29.16667 35.     ]
----------------------------------------


In [5]:
# ---------------------------------------------------------------------------------------------------------------------
# Longitudinal Mesh: Station at each bracing + Sub-divisions per girder length between bracings
# ---------------------------------------------------------------------------------------------------------------------
length_mesh = []
for i in range(len(brace_location)-1):
    x_start      = brace_location[i]
    x_end        = brace_location[i+1]
    panel_length = x_end - x_start
    divisions_len    = max(1, int(np.ceil(panel_length / length_mesh_size)))
    panel_mesh_len   = np.linspace(x_start, x_end, divisions_len + 1)
    length_mesh.extend(panel_mesh_len[:-1])
length_mesh.append(brace_location[-1])
length_mesh      = np.unique(np.round(length_mesh, 5))
num_length_mesh  = len(length_mesh)
print('\n----------------------------------------')
print(f'Longitudinal Stations       = {num_length_mesh} Stations')
print(length_mesh)
print(f'Min Longitudinal Mesh Size  = {np.min(np.diff(length_mesh)):.3f} m')
print(f'Max Longitudinal Mesh Size  = {np.max(np.diff(length_mesh)):.3f} m')
print('----------------------------------------')



----------------------------------------
Longitudinal Stations       = 145 Stations
[ 0.       0.24306  0.48611  0.72917  0.97222  1.21528  1.45833  1.70139
  1.94444  2.1875   2.43055  2.67361  2.91666  3.15972  3.40278  3.64583
  3.88889  4.13194  4.375    4.61805  4.86111  5.10416  5.34722  5.59027
  5.83333  6.07639  6.31944  6.5625   6.80555  7.04861  7.29166  7.53472
  7.77778  8.02083  8.26389  8.50694  8.75     8.99306  9.23611  9.47917
  9.72222  9.96528 10.20834 10.45139 10.69445 10.9375  11.18056 11.42361
 11.66667 11.90973 12.15278 12.39584 12.63889 12.88195 13.125   13.36806
 13.61111 13.85417 14.09722 14.34028 14.58334 14.82639 15.06945 15.3125
 15.55556 15.79861 16.04167 16.28472 16.52778 16.77083 17.01389 17.25694
 17.5     17.74306 17.98611 18.22917 18.47222 18.71528 18.95833 19.20139
 19.44444 19.6875  19.93055 20.17361 20.41667 20.65972 20.90278 21.14583
 21.38889 21.63194 21.875   22.11805 22.36111 22.60416 22.84722 23.09027
 23.33333 23.57639 23.81944 24.0625  24.

In [6]:
# ---------------------------------------------------------------------------------------------------------------------
# Transverse Mesh
# ---------------------------------------------------------------------------------------------------------------------
z1 = footpath_left
z2 = z1 + kerb_left
z3 = z2 + carriageway_1
z4 = z3 + median_width
z5 = z4 + carriageway_2
z6 = z5 + kerb_right
key_locations = [0.0, z1, z2, z3, z4, z5, z6, width]
key_locations.extend(girder_location)
key_locations = np.unique(np.round(key_locations, 5))
print('\n----------------------------------------')
print('Cross-section Key Locations')
print(key_locations)
print('----------------------------------------')

width_mesh = []
for i in range(len(key_locations)-1):
    z_start     = key_locations[i]
    z_end       = key_locations[i+1]
    panel_width = z_end - z_start
    divisions_wid   = max(1, int(np.ceil(panel_width / width_mesh_size)))
    panel_mesh_wid  = np.linspace(z_start, z_end, divisions_wid + 1)
    width_mesh.extend(panel_mesh_wid[:-1])
width_mesh.append(key_locations[-1])
width_mesh      = np.unique(np.round(width_mesh, 5))
num_width_mesh  = len(width_mesh)
print('\n----------------------------------------')
print(f'Transverse Stations     = {num_width_mesh} Stations')
print(width_mesh)
print(f'Min Width Mesh Size     = {np.min(np.diff(width_mesh)):.3f} m')
print(f'Max Width Mesh Size     = {np.max(np.diff(width_mesh)):.3f} m')
print('----------------------------------------')


----------------------------------------
Cross-section Key Locations
[ 0.    1.25  1.5   1.95  4.    6.45  6.75  7.05  9.5  11.55 12.   12.25
 13.5 ]
----------------------------------------

----------------------------------------
Transverse Stations     = 59 Stations
[ 0.       0.25     0.5      0.75     1.       1.25     1.5      1.725
  1.95     2.17778  2.40556  2.63333  2.86111  3.08889  3.31667  3.54444
  3.77222  4.       4.245    4.49     4.735    4.98     5.225    5.47
  5.715    5.96     6.205    6.45     6.6      6.75     6.9      7.05
  7.295    7.54     7.785    8.03     8.275    8.52     8.765    9.01
  9.255    9.5      9.72778  9.95556 10.18333 10.41111 10.63889 10.86667
 11.09444 11.32222 11.55    11.775   12.      12.25    12.5     12.75
 13.      13.25    13.5    ]
Min Width Mesh Size     = 0.150 m
Max Width Mesh Size     = 0.250 m
----------------------------------------


In [7]:
# ---------------------------------------------------------------------------------------------------------------------
# Deck Nodes
# ---------------------------------------------------------------------------------------------------------------------
DECK_BASE = 1000

deck_y      = 0.0 # Deck centroid elevation (m)
deck_nodes  = {}
for i, x in enumerate(length_mesh):
    for j, z in enumerate(width_mesh):
        node_tag = DECK_BASE + i*num_width_mesh + j
        ops.node(node_tag, float(x), float(deck_y), float(z))
        deck_nodes[(i,j)] = node_tag


In [8]:
# ---------------------------------------------------------------------------------------------------------------------
# Girder Nodes
# ---------------------------------------------------------------------------------------------------------------------
GIRDER_BASE = DECK_BASE + num_length_mesh*num_width_mesh 

girder_y        = -((girder_depth - ybar_girder) + deck_thickness/2) # Girder centroid elevation (m)
girder_nodes    = {}
for k, z in enumerate(girder_location):
    for i, x in enumerate(length_mesh):
        node_tag = GIRDER_BASE + k*num_length_mesh + i
        ops.node(node_tag, float(x), float(girder_y), float(z))
        girder_nodes[(k,i)] = node_tag


In [9]:
# ---------------------------------------------------------------------------------------------------------------------
# Bottom Nodes Connect Braces
# ---------------------------------------------------------------------------------------------------------------------
BOTTOM_BRACE_BASE = GIRDER_BASE + girder_num*num_length_mesh   

bottom_brace_y = girder_y - ybar_girder
bottom_brace_nodes = {}
for k, z in enumerate(girder_location):
    for n, x in enumerate(brace_location):
        node_tag = BOTTOM_BRACE_BASE + k*brace_num + n
        ops.node(node_tag, float(x), float(bottom_brace_y), float(z))
        bottom_brace_nodes[(k,n)] = node_tag

# ---------------------------------------------------------------------------------------------------------------------
# Top Nodes Connect Braces
# ---------------------------------------------------------------------------------------------------------------------
TOP_BRACE_BASE = BOTTOM_BRACE_BASE + girder_num*brace_num

top_brace_y = girder_y + (girder_depth - ybar_girder)   # = -deck_thickness/2, the girder's top flange / deck soffit level
top_brace_nodes = {}
for k, z in enumerate(girder_location):
    for n, x in enumerate(brace_location):
        node_tag = TOP_BRACE_BASE + k*brace_num + n
        ops.node(node_tag, float(x), float(top_brace_y), float(z))
        top_brace_nodes[(k,n)] = node_tag

# ---------------------------------------------------------------------------------------------------------------------
# K-Bracing Centre Nodes
# ---------------------------------------------------------------------------------------------------------------------
KBRACE_BASE = TOP_BRACE_BASE + girder_num*brace_num   

K_brace_nodes = {}
if brace_config.upper().startswith('K'):
    for n, x in enumerate(brace_location):
        for k in range(girder_num - 1):
            z_left  = girder_location[k]
            z_right = girder_location[k + 1]
            z_centre = 0.5 * (z_left + z_right)
            node_tag = KBRACE_BASE + n*(girder_num - 1) + k
            ops.node(node_tag, float(x), float(bottom_brace_y), float(z_centre))
            K_brace_nodes[(k, n)] = node_tag

print('\n----------------------------------------')
print(f'Deck Nodes   = {len(deck_nodes)}')
print(f'Girder Nodes = {len(girder_nodes)}')
print(f'Bottom Brace Nodes = {len(bottom_brace_nodes)}')
print(f'Top Brace Nodes      = {len(top_brace_nodes)}')
print(f'K-Brace Centre Nodes = {len(K_brace_nodes)}')
print(f'Total Nodes  = {len(deck_nodes) + len(girder_nodes) + len(bottom_brace_nodes) + len(top_brace_nodes) + len(K_brace_nodes)}')
print('----------------------------------------')



----------------------------------------
Deck Nodes   = 8555
Girder Nodes = 725
Bottom Brace Nodes = 35
Top Brace Nodes      = 35
K-Brace Centre Nodes = 0
Total Nodes  = 9350
----------------------------------------


In [10]:
# ---------------------------------------------------------------------------------------------------------------------
# Deck Elements
# ---------------------------------------------------------------------------------------------------------------------
DECK_ELE_BASE = 1000

ops.section('ElasticMembranePlateSection', 1, E_conc, Mu_conc, deck_thickness)
deck_elements   = {}
ele_tag         = DECK_ELE_BASE
for i in range(num_length_mesh-1):
    for j in range(num_width_mesh-1):
        n1 = deck_nodes[(i,j)]
        n2 = deck_nodes[(i+1,j)]
        n3 = deck_nodes[(i+1,j+1)]
        n4 = deck_nodes[(i,j+1)]
        ops.element('ShellMITC4', ele_tag, n1, n2, n3, n4, 1)
        deck_elements[(i,j)] = ele_tag
        ele_tag += 1

In [11]:
# ---------------------------------------------------------------------------------------------------------------------
# Gider Elements
# ---------------------------------------------------------------------------------------------------------------------
GIRDER_ELE_BASE = DECK_ELE_BASE + (num_length_mesh-1)*(num_width_mesh-1)   

ops.geomTransf('Linear', 1, 0, 1, 0)
girder_elements = {}
ele_tag         = GIRDER_ELE_BASE
for k in range(girder_num):
    for i in range(num_length_mesh-1):
        node_i = girder_nodes[(k,i)]
        node_j = girder_nodes[(k,i+1)]
        ops.element('elasticBeamColumn', ele_tag, node_i, node_j, A_girder, E_steel, G_steel, J_girder, Iyy_girder, Izz_girder, 1)
        girder_elements[(k,i)] = ele_tag
        ele_tag += 1

In [12]:
# ---------------------------------------------------------------------------------------------------------------------
# Deck-Girder Connection
# ---------------------------------------------------------------------------------------------------------------------
# Using Rigid Links
rigid_links = {}
girder_to_width_j = {}
for k, z in enumerate(girder_location):
    deck_j = np.where(np.isclose(width_mesh, z))[0][0]
    assert np.isclose(width_mesh[deck_j], z, atol=1e-4), f"Girder {k} at z={z} does not land on a width_mesh node"
    girder_to_width_j[k] = deck_j
    for i in range(num_length_mesh):
        girder_node = girder_nodes[(k, i)]
        deck_node   = deck_nodes[(i, deck_j)] 
        # ops.rigidLink('beam', deck_node, girder_node)  # deck = master, girder = slave
        ops.rigidLink('beam', girder_node, deck_node) # girder = master, deck = slave
        rigid_links[(k,i)] = (girder_node, deck_node)

# ---------------------------------------------------------------------------------------------------------------------
# Girder - Bottom Brace Node Connection
# ---------------------------------------------------------------------------------------------------------------------
# Using Rigid Links
rigid_links_bottom = {}
for k in range(girder_num):
    for n, x in enumerate(brace_location):
        girder_i    = np.where(np.isclose(length_mesh, x))[0][0]
        girder_node = girder_nodes[(k, girder_i)]
        brace_node  = bottom_brace_nodes[(k, n)]
        ops.rigidLink('beam', girder_node, brace_node)   # girder = master, brace node = slave
        rigid_links_bottom[(k,n)] = (girder_node, brace_node)

# ---------------------------------------------------------------------------------------------------------------------
# Girder - Top Brace Node Connection
# ---------------------------------------------------------------------------------------------------------------------
# Using Rigid Links
rigid_links_top = {}
for k in range(girder_num):
    for n, x in enumerate(brace_location):
        girder_i    = np.where(np.isclose(length_mesh, x))[0][0]
        girder_node = girder_nodes[(k, girder_i)]
        brace_node  = top_brace_nodes[(k, n)]
        ops.rigidLink('beam', girder_node, brace_node) # girder = master, brace node = slave
        rigid_links_top[(k,n)] = (girder_node, brace_node)


In [13]:
# ---------------------------------------------------------------------------------------------------------------------
# Bracing Elements
# ---------------------------------------------------------------------------------------------------------------------
ops.uniaxialMaterial("Elastic", 10, E_steel)
brace_elements  = {}
BRACE_ELE_BASE  = GIRDER_ELE_BASE + girder_num*(num_length_mesh-1)
ele_tag         = BRACE_ELE_BASE

if brace_config.upper().startswith('X'):
    for n, x in enumerate(brace_location):
        for k in range(girder_num-1):
            top_left        = top_brace_nodes[(k,n)]
            top_right       = top_brace_nodes[(k+1,n)]
            bottom_left     = bottom_brace_nodes[(k,n)]
            bottom_right    = bottom_brace_nodes[(k+1,n)]

            ops.element('corotTruss', ele_tag, top_left, bottom_right, A_brace_diagonal, 10)
            brace_elements[(k,n,'D1')] = ele_tag
            ele_tag += 1

            ops.element('corotTruss', ele_tag, top_right, bottom_left, A_brace_diagonal, 10)
            brace_elements[(k,n,'D2')] = ele_tag
            ele_tag += 1

            if brace_config.upper() in ['XT', 'XTB']:
                ops.element('corotTruss', ele_tag, top_left, top_right, A_brace_top, 10)
                brace_elements[(k,n,'TOP')] = ele_tag
                ele_tag += 1

            if brace_config.upper() in ['XB', 'XTB']:
                ops.element('corotTruss', ele_tag, bottom_left, bottom_right, A_brace_bottom, 10)
                brace_elements[(k,n,'BOT')] = ele_tag
                ele_tag += 1

elif brace_config.upper().startswith('K'):
    for n, x in enumerate(brace_location):
        for k in range(girder_num-1):
            top_left        = top_brace_nodes[(k,n)]
            top_right       = top_brace_nodes[(k+1,n)]
            bottom_left     = bottom_brace_nodes[(k,n)]
            bottom_right    = bottom_brace_nodes[(k+1,n)]
            centre          = K_brace_nodes[(k,n)]

            ops.element('corotTruss', ele_tag, top_left, centre, A_brace_diagonal, 10)
            brace_elements[(k,n,'KL')] = ele_tag
            ele_tag += 1

            ops.element('corotTruss', ele_tag, top_right, centre, A_brace_diagonal, 10)
            brace_elements[(k,n,'KR')] = ele_tag
            ele_tag += 1

            ops.element('corotTruss', ele_tag, bottom_left, centre, A_brace_diagonal, 10)
            brace_elements[(k,n,'BL')] = ele_tag
            ele_tag += 1

            ops.element('corotTruss', ele_tag, centre, bottom_right, A_brace_diagonal, 10)
            brace_elements[(k,n,'BR')] = ele_tag
            ele_tag += 1

            if brace_config.upper() == 'KT':
                ops.element('corotTruss', ele_tag, top_left, top_right, A_brace_top, 10)
                brace_elements[(k,n,'TOP')] = ele_tag
                ele_tag += 1

print('\n----------------------------------------')
print(f'Deck Shell Elements         = {len(deck_elements)}')
print(f'Girder Elements             = {len(girder_elements)}')
print(f'Rigid Links (Deck-Girder)   = {len(rigid_links)}')
print(f'Rigid Links (Top Brace)     = {len(rigid_links_top)}')
print(f'Rigid Links (Bottom Brace)  = {len(rigid_links_bottom)}')
print(f'Bracing Configuration       = {brace_config}')
print(f'Brace Elements              = {len(brace_elements)}')
print('----------------------------------------')



----------------------------------------
Deck Shell Elements         = 8352
Girder Elements             = 720
Rigid Links (Deck-Girder)   = 725
Rigid Links (Top Brace)     = 35
Rigid Links (Bottom Brace)  = 35
Bracing Configuration       = X
Brace Elements              = 56
----------------------------------------


In [108]:

# =====================================================================================================================
# LOADS
# =====================================================================================================================
# ---------------------------------------------------------------------------------------------------------------------
# Support Conditions
# ---------------------------------------------------------------------------------------------------------------------
for k in range(girder_num):
    left_node  = girder_nodes[(k, 0)]
    right_node = girder_nodes[(k, num_length_mesh-1)]
    ops.fix(left_node,  1, 1, 1, 0, 0, 0)
    ops.fix(right_node, 0, 1, 1, 0, 0, 0)

for node_tag in K_brace_nodes.values():
    ops.fix(node_tag, 1, 0, 0, 1, 1, 1)  

# ---------------------------------------------------------------------------------------------------------------------
# Dead Loads
# ---------------------------------------------------------------------------------------------------------------------
ops.timeSeries('Linear', 1)
ops.pattern('Plain', 1, 1)

q_deck      = Rho_conc * deck_thickness
q_asphalt   = Rho_asphalt * asphalt_thickness
q_footpath  = Rho_footpath * footpath_thickness
q_kerb      = Rho_kerb * kerb_thickness
q_median    = Rho_median * median_thickness

z_fp_left_end      = footpath_left
z_kerb_left_end    = z_fp_left_end + kerb_left
z_cw1_end          = z_kerb_left_end + carriageway_1
z_median_end       = z_cw1_end + median_width
z_cw2_end          = z_median_end + carriageway_2
z_kerb_right_end   = z_cw2_end + kerb_right

total_base_deck_load = 0.0
total_asphalt_load = 0.0
total_footpath_left_load = 0.0
total_footpath_right_load = 0.0
total_kerb_left_load = 0.0
total_kerb_right_load = 0.0
total_median_load = 0.0
total_deck_load = 0.0

for i in range(num_length_mesh):
    for j in range(num_width_mesh):
        node = deck_nodes[(i,j)]
        z = width_mesh[j]

        # Longitudinal Tributary Length
        if i == 0:
            tx = (length_mesh[1] - length_mesh[0]) / 2
        elif i == num_length_mesh - 1:
            tx = (length_mesh[-1] - length_mesh[-2]) / 2
        else:
            tx = (length_mesh[i+1] - length_mesh[i-1]) / 2

        # Transverse Tributary Width
        if j == 0:
            tz = (width_mesh[1] - width_mesh[0]) / 2
        elif j == num_width_mesh - 1:
            tz = (width_mesh[-1] - width_mesh[-2]) / 2
        else:
            tz = (width_mesh[j+1] - width_mesh[j-1]) / 2
        tributary_area = tx * tz
        
        # Deck Self Weight
        P_total = q_deck * tributary_area
        total_base_deck_load += q_deck * tributary_area
         
         # Asphalt Load
        if median_width > 0:
            is_asphalt = (((z > z_kerb_left_end) and (z < z_cw1_end)) or ((z > z_median_end) and (z < z_cw2_end)))
        else:
            # No median: asphalt is continuous across both carriageways
            is_asphalt = (z > z_kerb_left_end) and (z < z_cw2_end)

        if is_asphalt:
            P_total += q_asphalt * tributary_area
            total_asphalt_load += q_asphalt * tributary_area

         # Footpath Load
        if footpath_left > 0 and z <= z_fp_left_end:
            P_total += q_footpath * tributary_area
            total_footpath_left_load += q_footpath * tributary_area

        if footpath_right > 0 and z >= z_kerb_right_end:
            P_total += q_footpath * tributary_area
            total_footpath_right_load += q_footpath * tributary_area            

        # Kerb Load
        if kerb_left > 0 and (z > z_fp_left_end) and (z <= z_kerb_left_end):
            P_total += q_kerb * tributary_area
            total_kerb_left_load += q_kerb * tributary_area 

        if kerb_right > 0 and (z >= z_cw2_end) and (z < z_kerb_right_end):
            P_total += q_kerb * tributary_area
            total_kerb_right_load += q_kerb * tributary_area

        # Median Load
        if median_width > 0 and (z >= z_cw1_end) and (z <= z_median_end):
            P_total += q_median * tributary_area
            total_median_load += q_median * tributary_area 

        ops.load(node, 0.0, -P_total, 0.0, 0.0, 0.0, 0.0)
        total_deck_load += P_total

# Girder Self Weight
q_girder = Rho_steel * A_girder
for ele in girder_elements.values():
    ops.eleLoad('-ele', ele, '-type', '-beamUniform', 0.0, -q_girder)

# Bracing Self Weight
q_brace = 0.0
for (k, n, label), ele in brace_elements.items():
    if label == 'TOP':
        A_ele = A_brace_top
    elif label == 'BOT':
        A_ele = A_brace_bottom
    else:  # D1, D2, KL, KR, BL, BR - all diagonals
        A_ele = A_brace_diagonal

    n1, n2      = ops.eleNodes(ele)
    x1, y1, z1  = ops.nodeCoord(n1)
    x2, y2, z2  = ops.nodeCoord(n2)
    L_brace     = ((x2-x1)**2 + (y2-y1)**2 + (z2-z1)**2)**0.5
    W_brace     = A_ele * Rho_steel * L_brace
    ops.load(n1, 0.0, -W_brace/2, 0.0, 0.0, 0.0, 0.0)
    ops.load(n2, 0.0, -W_brace/2, 0.0, 0.0, 0.0, 0.0)
    q_brace     += W_brace

expected_total_DL = total_deck_load + q_girder * length * girder_num + q_brace

print('\n----------------------------------------')
print('Dead Loads Applied')
print(f'Deck Load           = {q_deck:.3f} kN/m^2')
print(f'Asphalt Load        = {q_asphalt:.3f} kN/m^2')
print(f'Footpath Load       = {q_footpath:.3f} kN/m^2')
print(f'Kerb Load           = {q_kerb:.3f} kN/m^2')
print(f'Median Load         = {q_median:.3f} kN/m^2')
print(f'Girder Self Weight  = {q_girder:.3f} kN/m')
print('----------------------------------------')


----------------------------------------
Dead Loads Applied
Deck Load           = 5.750 kN/m^2
Asphalt Load        = 1.650 kN/m^2
Footpath Load       = 3.600 kN/m^2
Kerb Load           = 7.200 kN/m^2
Median Load         = 6.000 kN/m^2
Girder Self Weight  = 5.428 kN/m
----------------------------------------


In [ ]:
# =====================================================================================================================
# DEAD LOAD ANALYSIS
# =====================================================================================================================
ops.wipeAnalysis()
ops.system('BandGeneral')
ops.numberer('RCM')
ops.constraints('Lagrange')
# ops.constraints('Transformation')
ops.test('NormDispIncr', 1.0e-8, 20)
ops.algorithm('Newton')
ops.integrator('LoadControl', 1.0)
ops.analysis('Static')

ok = ops.analyze(1)


In [ ]:
# =====================================================================================================================
# LOAD SUMMARY
# =====================================================================================================================
ops.reactions()

print('\n' + '='*80)
print('SUPPROT REACTIONS')
print('='*80)

Rx_total = 0.0
Ry_total = 0.0
Rz_total = 0.0

for k in range(girder_num):

    # Left support
    left_node = girder_nodes[(k, 0)]
    RxL = ops.nodeReaction(left_node, 1)
    RyL = ops.nodeReaction(left_node, 2)
    RzL = ops.nodeReaction(left_node, 3)

    # Right support
    right_node = girder_nodes[(k, num_length_mesh-1)]
    RxR = ops.nodeReaction(right_node, 1)
    RyR = ops.nodeReaction(right_node, 2)
    RzR = ops.nodeReaction(right_node, 3)

    # Sum reactions
    Rx_total += RxL + RxR
    Ry_total += RyL + RyR
    Rz_total += RzL + RzR

    # Print in one line
    print(f'Girder {k+1}: left Rx={RxL:8.3f}, right Rx={RxR:8.3f}')
    print(f'Girder {k+1}: left Ry={RyL:8.3f}, right Ry={RyR:8.3f}')
    print(f'Girder {k+1}: left Rz={RzL:8.3f}, right Rz={RzR:8.3f}')

print()
print(f'Total Base Deck Load        = {total_base_deck_load:12.2f} kN')
print(f'Total Asphalt Load          = {total_asphalt_load:12.3f} kN')
print(f'Total Footpath Load (L)     = {total_footpath_left_load:12.3f} kN')
print(f'Total Footpath Load (R)     = {total_footpath_right_load:12.3f} kN')
print(f'Total Kerb Load (L)         = {total_kerb_left_load:12.3f} kN')
print(f'Total Kerb Load (R)         = {total_kerb_right_load:12.3f} kN')
print(f'Total Median Load           = {total_median_load:12.3f} kN')
print(f'Total Dead Load on Deck     = {total_deck_load:12.3f} kN')
print(f'Total Girder self-Weight    = {q_girder*length*girder_num:12.3f} kN')
print(f'Total Bracing Self-Weight   = {q_brace:12.3f} kN')
print(f'Expected Total Dead load    = {expected_total_DL:12.3f} kN')
print(f'Total Rx                    = {Rx_total:12.3f} kN')
print(f'Total Ry                    = {Ry_total:12.3f} kN')
print(f'Total Rz                    = {Rz_total:12.3f} kN')
